<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h1>Lab 01 · Real Delta Bronze</h1><p>SDA-DSC-214 · Meaad Al-Marri</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h1>اللاب 01 · Bronze فعلية باستخدام Delta</h1><p>SDA-DSC-214 · ميعاد المري</p></td></tr></tbody></table>

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Goal and status</h2><p><strong>ENGINE_NOT_EXECUTED</strong> — source code is available; engine execution is pending. Read <a href="../../labs/lab01/WALKTHROUGH.md">the walkthrough</a>. Expected counts are teaching assertions, not saved runtime results.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>الهدف والحالة</h2><p><strong>ENGINE_NOT_EXECUTED</strong> — الكود موجود وتشغيل المحرك ينتظر التحقق. اقرأ <a href="../../labs/lab01/WALKTHROUGH.md">الشرح المتدرج</a>. الأعداد المتوقعة فحوص تعليمية وليست نتائج تشغيل محفوظة.</p></td></tr></tbody></table>

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Setup · report the actual environment</h2><p>Only inspect dependencies here. This cell does not install packages or start Spark.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>الإعداد · اعرض البيئة الفعلية</h2><p>افحص المتطلبات فقط هنا. لا تثبت هذه الخلية حزمًا ولا تبدأ Spark.</p></td></tr></tbody></table>

In [ ]:
from pathlib import Path
import sys, json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "course.json").is_file() and (p / "src/masar").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open the notebook from within the complete course repository")
sys.path.insert(0, str(ROOT / "src"))
SOURCE = ROOT / "data/masar-small-v1"
from masar.runtime import inspect_environment, require_environment, start_spark
print(json.dumps(inspect_environment(), indent=2))

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Start a clean, bounded run</h2><p>Fail clearly if dependencies are missing; never switch engines silently. Existing work is retained.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>ابدأ تشغيلًا جديدًا محدود النطاق</h2><p>توقف بوضوح عند نقص المتطلبات دون تبديل المحرك بصمت. تبقى الأعمال السابقة محفوظة.</p></td></tr></tbody></table>

In [ ]:
require_environment()
from masar.workspace import new_workspace, require_fixed_dataset, write_json, workspace_path
require_fixed_dataset(SOURCE)
WORK = new_workspace(ROOT, "day01_bronze")
print("Workspace:", WORK.relative_to(ROOT))

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Read one source</h2><p>CSV strings remain unchanged. The count action starts computation.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>اقرأ مصدرًا واحدًا</h2><p>تبقى نصوص CSV كما هي. يطلق فعل العد الحساب الفعلي.</p></td></tr></tbody></table>

In [ ]:
from masar.bronze import raw_frame, ingest_feed, verify_bronze
# Functions are imported without starting Spark. The next cell executes the lab.
print("Bronze source code loaded; engine has not started yet.")

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Load, replay, verify and retain evidence</h2><p>The <code>try/finally</code> always stops the session after this block, even on failure. Reuse of a committed batch id is rejected. See <a href="../../src/masar/bronze.py">shared source</a>; this is not a placeholder or mock engine.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>استوعب وأعد الوصول وتحقق واحفظ الدليل</h2><p>توقف <code>try/finally</code> الجلسة بعد هذه الكتلة حتى عند الفشل. تُرفض إعادة استخدام معرف دفعة محفوظ. راجع <a href="../../src/masar/bronze.py">المصدر المشترك</a>؛ ليست هذه محاكاة للمحرك أو موضعًا فارغًا.</p></td></tr></tbody></table>

In [ ]:
from masar.workspace import record_bronze_success
spark = start_spark(WORK)
try:
    print("Spark:", spark.version)
    raw_trips = raw_frame(spark, SOURCE, "trips")
    raw_trips.printSchema()
    raw_trips.show(3, truncate=False)
    print("Source trips:", raw_trips.count())
    for feed in ("trips", "drivers", "gps_events"):
        print(ingest_feed(spark, SOURCE, WORK, feed, "base_001"))
    print(ingest_feed(spark, SOURCE, WORK, "trips", "replay_002"))
    report = verify_bronze(spark, SOURCE, WORK)
    record_bronze_success(ROOT, WORK)
    print(json.dumps({"counts": report["counts"], "checks": report["checks"]}, indent=2))
    print("Retained:", (WORK / "reports/bronze.json").relative_to(ROOT))
finally:
    spark.stop()

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Checks and next step</h2><p>A successful scenario has 144 Bronze trip deliveries but 72 distinct trip ids. Read <a href="../../day01/PRACTICE.md">the reasoning prompts</a>, complete Lab 01 notes, then use the same successful workspace in <a href="04_spark_scan.ipynb">Lab 02</a>. No generated table is automatically committed to Git.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>الفحوص والخطوة التالية</h2><p>يحتوي السيناريو الناجح 144 سجل وصول في Bronze و72 معرف رحلة فريدًا. اقرأ <a href="../../day01/PRACTICE.md">أسئلة التفكير</a> وأكمل ملاحظات اللاب 01، ثم استخدم مساحة العمل الناجحة نفسها في <a href="04_spark_scan.ipynb">اللاب 02</a>. لا يضاف أي جدول مولد إلى Git تلقائيًا.</p></td></tr></tbody></table>